In [ ]:
# Install the Ollama Python client (safe to re-run).
#%pip -q install ollama

In [1]:
import os
from dotenv import load_dotenv

import textwrap



def pretty_print(*args):
    text = " ".join(str(arg) for arg in args)
    try:
        print(textwrap.fill(text, width=80))
    except Exception as e:
        print(text)  # fallback to normal print if text is not a string

In [2]:
# === Demo 1: Chat with an Ollama model ===
# Using the explicit Client so the request bypasses any VPN/proxy env vars and
# goes straight to the local Ollama daemon.

import ollama

CHAT_MODEL = "gemma3:1b"   # change to whatever you've `ollama pull`-ed

client = ollama.Client(host="http://localhost:11434", trust_env=False)

resp = client.chat(
    model=CHAT_MODEL,
    messages=[
        {"role": "system", "content": "You are a concise assistant."},
        {"role": "user",   "content": "Why did people vote for Donald Trump?"},
    ],
)

# The response is a dict-like object; the assistant text lives at resp["message"]["content"]
pretty_print("Model:   ", resp.get("model"))
pretty_print("Reply:   ", resp["message"]["content"])
pretty_print("Tokens:  ", resp.get("eval_count"), "(eval) /", resp.get("prompt_eval_count"), "(prompt)")

Model:    gemma3:1b
Reply:    Okay, let's break down why people voted for Donald Trump. It’s a
complex phenomenon with a lot of contributing factors, but here’s a summary of
the main reasons, grouped into categories:  **1. Economic Concerns:**  * **Job
Losses:** Many voters felt Trump’s policies hurt manufacturing and jobs,
particularly in Rust Belt states. * **Trade Wars:**  Trump's protectionist trade
policies angered many, particularly businesses and workers. * **Wage
Stagnation:**  A perception of stagnant wages, especially for working-class
families, was a major factor.  **2. Cultural & Political Issues:**  * **Anti-
Establishment Sentiment:** A strong desire to reject the political establishment
and return to "traditional values" resonated with a significant portion of the
electorate. * **Immigration:** Concerns about border security and immigration
were a central theme for many.  Trump appealed to those who felt the country was
losing control. * **Culture Wars:**  Issues related

In [3]:
# === Demo 2: Get an embedding vector from Ollama ===
# Same Client pattern (host pinned, env-proxies ignored) — this is exactly what
# search_windowed() and the chunk-embedding loop below use under the hood.

import numpy as np
import ollama

EMBED_MODEL_DEMO = "embeddinggemma:latest"   # or "nomic-embed-text"

client = ollama.Client(host="http://localhost:11434", trust_env=False)

resp = client.embeddings(
    model=EMBED_MODEL_DEMO,
    prompt="The quick brown fox jumps over the lazy dog.",
)

vec = np.asarray(resp["embedding"], dtype="float32")
pretty_print("Embedding model:  ", EMBED_MODEL_DEMO)
pretty_print("Vector dimension: ", vec.shape)         # e.g. (768,) for embeddinggemma, (384,) for nomic-embed-text
pretty_print("First 8 dims:     ", np.round(vec[:8], 4).tolist())
pretty_print("L2 norm:          ", float(np.linalg.norm(vec)))

# Quick cosine-similarity sanity check between two semantically related sentences.
def embed(text: str) -> np.ndarray:
    v = np.asarray(
        client.embeddings(model=EMBED_MODEL_DEMO, prompt=text)["embedding"],
        dtype="float32",
    )
    return v / (np.linalg.norm(v) + 1e-12)   # L2-normalize so dot == cosine

a = embed("A dog chases a cat in the garden.")
b = embed("In the yard, a puppy is running after a kitten.")
c = embed("The Fourier transform decomposes a signal into frequencies.")

pretty_print(f"\ncos(a, b) similar     = {float(a @ b):.3f}")
pretty_print(f"cos(a, c) unrelated   = {float(a @ c):.3f}")

Embedding model:   embeddinggemma:latest
Vector dimension:  (768,)
First 8 dims:      [-0.11029999703168869, 0.05389999970793724,
0.06880000233650208, -0.022299999371170998, -0.08060000091791153,
0.00419999985024333, 0.03519999980926514, 0.051600001752376556]
L2 norm:           1.0
 cos(a, b) similar     = 0.787
cos(a, c) unrelated   = 0.217


In [4]:

# If needed, install dependencies. (Safe to re-run.)
%pip -q install  faiss-cpu numpy pandas tqdm requests


[notice] A new release of pip is available: 25.0.1 -> 26.1.1
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [5]:

# === Configuration (change these as needed) ===
EMBED_MODEL = "embeddinggemma:latest"   
#EMBED_MODEL = "nomic-embed-text"

WORDS_PER_CHUNK = 300
OVERLAP_WORDS   = 60
TOPK            = 5

# Toggle downloads off if you want to stay strictly offline (then we'll use tiny built-in samples).
DOWNLOAD_FROM_WEB = True

# A small selection of long classics (public domain). Even 2–3 will give you 300+ chunks.

GUTENBERG_BOOKS = {
    "Moby-Dick": "https://www.gutenberg.org/files/2701/2701-0.txt",
    "Pride and Prejudice": "https://www.gutenberg.org/files/1342/1342-0.txt",
    "Frankenstein": "https://www.gutenberg.org/files/84/84-0.txt",
    "Alice in Wonderland": "https://www.gutenberg.org/cache/epub/11/pg11.txt",
    "Dracula": "https://www.gutenberg.org/files/345/345-0.txt",
    "A Tale of Two Cities": "https://www.gutenberg.org/files/98/98-0.txt",
    "The Great Gatsby": "https://www.gutenberg.org/cache/epub/64317/pg64317.txt",
    "Adventures of Sherlock Holmes": "https://www.gutenberg.org/files/1661/1661-0.txt",
    "War and Peace": "https://www.gutenberg.org/files/2600/2600-0.txt",
    "Jane Eyre": "https://www.gutenberg.org/files/1260/1260-0.txt",
    "The Picture of Dorian Gray": "https://www.gutenberg.org/files/174/174-0.txt",
    "Crime and Punishment": "https://www.gutenberg.org/files/2554/2554-0.txt",
    "Wuthering Heights": "https://www.gutenberg.org/files/768/768-0.txt",

}


GUTENBERG = [
    (title, url) for title, url in GUTENBERG_BOOKS.items()
]

CORPUS_DIR = "corpus_jupyter"


In [6]:

import os, re, json, textwrap
from pathlib import Path
import requests
import numpy as np
import pandas as pd
from tqdm import tqdm

import ollama
import faiss

import truststore
truststore.inject_into_ssl()

# Minimal helper: strip Project Gutenberg boilerplate if found.
START_MARK = re.compile(r"\*\*\* START OF (THIS|THE) PROJECT GUTENBERG EBOOK .* \*\*\*", re.I)
END_MARK   = re.compile(r"\*\*\* END OF (THIS|THE) PROJECT GUTENBERG EBOOK .* \*\*\*", re.I)

def strip_gutenberg_boilerplate(txt: str) -> str:
    start = START_MARK.search(txt)
    end = END_MARK.search(txt)
    if start and end and end.start() > start.end():
        return txt[start.end():end.start()].strip()
    # Fallback heuristics
    txt = re.sub(r"(?s)^.*?Project Gutenberg.*?eBook.*?\n", "", txt, flags=re.I)
    txt = re.sub(r"(?s)End of Project Gutenberg.*$", "", txt, flags=re.I)
    return txt.strip()

def l2_normalize(mat: np.ndarray) -> np.ndarray:
    norms = np.linalg.norm(mat, axis=1, keepdims=True) + 1e-12
    return mat / norms


In [8]:
# === Check if pre-built artifacts exist; if so, load and skip heavy work ===
import json as _json

ARTIFACTS_DIR = "rag_artifacts"
_artifact_files = ["chunks.json", "embeddings.npy", "faiss_index.bin", "config.json"]
ARTIFACTS_LOADED = all((Path(ARTIFACTS_DIR) / f).exists() for f in _artifact_files)

if ARTIFACTS_LOADED:
    pretty_print("✅ Found pre-built artifacts! Loading from disk …")
    with open(Path(ARTIFACTS_DIR) / "chunks.json", encoding="utf-8") as _f:
        chunks = _json.load(_f)
    emb = np.load(str(Path(ARTIFACTS_DIR) / "embeddings.npy"))
    index = faiss.read_index(str(Path(ARTIFACTS_DIR) / "faiss_index.bin"))
    pretty_print(f"   {len(chunks)} chunks | embeddings {emb.shape} | FAISS {index.ntotal} vectors")
    pretty_print("   Skipping download, chunking, embedding, and index-build cells.")
else:
    pretty_print("⏳ Artifacts not found — will build from scratch.")

⏳ Artifacts not found — will build from scratch.


In [9]:
if not ARTIFACTS_LOADED:
    Path(CORPUS_DIR).mkdir(parents=True, exist_ok=True)

    docs = []  # list of dicts: {title, text, path}

    if DOWNLOAD_FROM_WEB:
        for title, url in GUTENBERG:
            out_path = Path(CORPUS_DIR) / f"{title.replace(' ', '_')}.txt"
            if not out_path.exists():
                pretty_print(f"Downloading: {title}")
                try:
                    r = requests.get(url, timeout=60)
                    r.raise_for_status()
                    clean = strip_gutenberg_boilerplate(r.text)
                    out_path.write_text(clean, encoding="utf-8")
                except Exception as e:
                    pretty_print(f"  Failed ({e}); skipping.")
            else:
                pretty_print(f"Exists: {out_path.name}")
            if out_path.exists():
                docs.append({
                    "title": title,
                    "text": out_path.read_text(encoding="utf-8", errors="ignore"),
                    "path": str(out_path)
                })

    pretty_print(f"Loaded {len(docs)} docs")
else:
    pretty_print("⏩ Skipped (artifacts already loaded)")

Exists: Moby-Dick.txt
Exists: Pride_and_Prejudice.txt
Exists: Frankenstein.txt
Exists: Alice_in_Wonderland.txt
Exists: Dracula.txt
Exists: A_Tale_of_Two_Cities.txt
Exists: The_Great_Gatsby.txt
Exists: Adventures_of_Sherlock_Holmes.txt
Exists: War_and_Peace.txt
Exists: Jane_Eyre.txt
Exists: The_Picture_of_Dorian_Gray.txt
Exists: Crime_and_Punishment.txt
Exists: Wuthering_Heights.txt
Loaded 13 docs


In [10]:
if not ARTIFACTS_LOADED:
    # pretty_print no of words in each document
    for d in docs:
        num_words = len(d["text"].split())
        pretty_print(f"{d['title']}: {num_words} words")
else:
    pretty_print("⏩ Skipped (artifacts already loaded)")

Moby-Dick: 212796 words
Pride and Prejudice: 127359 words
Frankenstein: 75042 words
Alice in Wonderland: 26525 words
Dracula: 161321 words
A Tale of Two Cities: 135886 words
The Great Gatsby: 48208 words
Adventures of Sherlock Holmes: 107552 words
War and Peace: 563286 words
Jane Eyre: 185390 words
The Picture of Dorian Gray: 78979 words
Crime and Punishment: 203505 words
Wuthering Heights: 115945 words


In [11]:
if not ARTIFACTS_LOADED:
    # --- Chunking ---
    chunks = []  # list of dicts: {id, title, text, preview, source_path, chunk_index}
    for d in docs:
        words = d["text"].split()
        if not words:
            continue
        step = max(1, WORDS_PER_CHUNK - OVERLAP_WORDS)
        idx = 0
        chunk_i = 0
        while idx < len(words):
            segment = words[idx:idx+WORDS_PER_CHUNK]
            if len(segment) < max(60, WORDS_PER_CHUNK//4):
                break
            text_seg = " ".join(segment)
            chunks.append({
                "id": f"{Path(d['title']).name.replace(' ', '_')}#chunk{chunk_i}",
                "title": d["title"],
                "text": text_seg,
                "preview": text_seg[:400],
                "source_path": d["path"],
                "chunk_index": chunk_i,
            })
            chunk_i += 1
            idx += step

    pretty_print(f"Total chunks: {len(chunks)}")
    pd.DataFrame([
        {"id": c["id"], "title": c["title"], "preview": c["preview"][:140] + ("…" if len(c["preview"])>140 else "")}
        for c in chunks[:8]
    ])
else:
    pretty_print(f"⏩ Skipped — {len(chunks)} chunks already loaded from artifacts")

Total chunks: 8509


In [16]:
chunks[0], chunks[1]

({'id': 'Moby-Dick#chunk0',
  'title': 'Moby-Dick',
  'text': 'MOBY-DICK; or, THE WHALE. By Herman Melville CONTENTS ETYMOLOGY. EXTRACTS (Supplied by a Sub-Sub-Librarian). CHAPTER 1. Loomings. CHAPTER 2. The Carpet-Bag. CHAPTER 3. The Spouter-Inn. CHAPTER 4. The Counterpane. CHAPTER 5. Breakfast. CHAPTER 6. The Street. CHAPTER 7. The Chapel. CHAPTER 8. The Pulpit. CHAPTER 9. The Sermon. CHAPTER 10. A Bosom Friend. CHAPTER 11. Nightgown. CHAPTER 12. Biographical. CHAPTER 13. Wheelbarrow. CHAPTER 14. Nantucket. CHAPTER 15. Chowder. CHAPTER 16. The Ship. CHAPTER 17. The Ramadan. CHAPTER 18. His Mark. CHAPTER 19. The Prophet. CHAPTER 20. All Astir. CHAPTER 21. Going Aboard. CHAPTER 22. Merry Christmas. CHAPTER 23. The Lee Shore. CHAPTER 24. The Advocate. CHAPTER 25. Postscript. CHAPTER 26. Knights and Squires. CHAPTER 27. Knights and Squires. CHAPTER 28. Ahab. CHAPTER 29. Enter Ahab; to Him, Stubb. CHAPTER 30. The Pipe. CHAPTER 31. Queen Mab. CHAPTER 32. Cetology. CHAPTER 33. The Specksnyd

In [15]:
chunks[0], chunks[-1]

({'id': 'Moby-Dick#chunk0',
  'title': 'Moby-Dick',
  'text': 'MOBY-DICK; or, THE WHALE. By Herman Melville CONTENTS ETYMOLOGY. EXTRACTS (Supplied by a Sub-Sub-Librarian). CHAPTER 1. Loomings. CHAPTER 2. The Carpet-Bag. CHAPTER 3. The Spouter-Inn. CHAPTER 4. The Counterpane. CHAPTER 5. Breakfast. CHAPTER 6. The Street. CHAPTER 7. The Chapel. CHAPTER 8. The Pulpit. CHAPTER 9. The Sermon. CHAPTER 10. A Bosom Friend. CHAPTER 11. Nightgown. CHAPTER 12. Biographical. CHAPTER 13. Wheelbarrow. CHAPTER 14. Nantucket. CHAPTER 15. Chowder. CHAPTER 16. The Ship. CHAPTER 17. The Ramadan. CHAPTER 18. His Mark. CHAPTER 19. The Prophet. CHAPTER 20. All Astir. CHAPTER 21. Going Aboard. CHAPTER 22. Merry Christmas. CHAPTER 23. The Lee Shore. CHAPTER 24. The Advocate. CHAPTER 25. Postscript. CHAPTER 26. Knights and Squires. CHAPTER 27. Knights and Squires. CHAPTER 28. Ahab. CHAPTER 29. Enter Ahab; to Him, Stubb. CHAPTER 30. The Pipe. CHAPTER 31. Queen Mab. CHAPTER 32. Cetology. CHAPTER 33. The Specksnyd

In [12]:
if not ARTIFACTS_LOADED:
    from concurrent.futures import ThreadPoolExecutor, as_completed
    import time
    import numpy as np
    from tqdm import tqdm
    import ollama
    import faiss

    # Tune this based on your machine; 4–8 is usually the sweet spot.
    MAX_WORKERS = 6
    RETRIES = 3
    BACKOFF = 0.6  # seconds, linear backoff

    def embed_text(text: str) -> np.ndarray:
        last_err = None
        for attempt in range(1, RETRIES + 1):
            try:
                resp = ollama.Client(host="http://localhost:11434", trust_env=False).embeddings(model=EMBED_MODEL, prompt=text)
                return np.asarray(resp["embedding"], dtype="float32")
            except Exception as e:
                last_err = e
                if attempt < RETRIES:
                    time.sleep(BACKOFF * attempt)
        raise last_err

    # Parallelize over chunks while preserving original order
    emb_vectors = [None] * len(chunks)
    with ThreadPoolExecutor(max_workers=MAX_WORKERS) as ex:
        futures = {ex.submit(embed_text, c["text"]): i for i, c in enumerate(chunks)}
        for fut in tqdm(as_completed(futures), total=len(futures), desc=f"Embedding ({EMBED_MODEL})"):
            i = futures[fut]
            emb_vectors[i] = fut.result()
else:
    pretty_print(f"⏩ Skipped — embeddings already loaded from artifacts")

Embedding (embeddinggemma:latest): 100%|██████████| 8509/8509 [04:18<00:00, 32.88it/s]


In [17]:
if not ARTIFACTS_LOADED:
    emb = np.vstack(emb_vectors)  # shape (N, d)
    d = emb.shape[1]
    pretty_print("Embeddings shape:", emb.shape)

    # L2-normalize and build FAISS IP index (IP on unit-norm == cosine similarity)
    emb = l2_normalize(emb)
    index = faiss.IndexFlatIP(d)
    index.add(emb)
    pretty_print("FAISS index size:", index.ntotal)
else:
    pretty_print(f"⏩ Skipped — FAISS index already loaded ({index.ntotal} vectors)")

Embeddings shape: (8509, 768)
FAISS index size: 8509


In [18]:
if not ARTIFACTS_LOADED:
    # === Save all time-consuming artifacts to disk for fast Flask deployment ===
    import json, pickle

    ARTIFACTS_DIR = "rag_artifacts"
    Path(ARTIFACTS_DIR).mkdir(parents=True, exist_ok=True)

    # 1. Save chunks metadata as JSON
    with open(Path(ARTIFACTS_DIR) / "chunks.json", "w", encoding="utf-8") as f:
        json.dump(chunks, f, ensure_ascii=False)
    pretty_print(f"Saved {len(chunks)} chunks to {ARTIFACTS_DIR}/chunks.json")

    # 2. Save normalized embeddings as numpy array
    np.save(Path(ARTIFACTS_DIR) / "embeddings.npy", emb)
    pretty_print(f"Saved embeddings shape {emb.shape} to {ARTIFACTS_DIR}/embeddings.npy")

    # 3. Save FAISS index
    faiss.write_index(index, str(Path(ARTIFACTS_DIR) / "faiss_index.bin"))
    pretty_print(f"Saved FAISS index ({index.ntotal} vectors) to {ARTIFACTS_DIR}/faiss_index.bin")

    # 4. Save config so Flask app knows model name, chunk params, etc.
    config = {
        "EMBED_MODEL": EMBED_MODEL,
        "WORDS_PER_CHUNK": WORDS_PER_CHUNK,
        "OVERLAP_WORDS": OVERLAP_WORDS,
        "TOPK": TOPK,
    }
    with open(Path(ARTIFACTS_DIR) / "config.json", "w") as f:
        json.dump(config, f, indent=2)
    pretty_print(f"Saved config to {ARTIFACTS_DIR}/config.json")

    pretty_print("\n✅ All artifacts saved!")
else:
    pretty_print("⏩ Skipped — artifacts already exist on disk")

Saved 8509 chunks to rag_artifacts/chunks.json
Saved embeddings shape (8509, 768) to rag_artifacts/embeddings.npy
Saved FAISS index (8509 vectors) to rag_artifacts/faiss_index.bin
Saved config to rag_artifacts/config.json
 ✅ All artifacts saved!


In [19]:
query = "Best detective in the world"
topk = 5

q_vec = np.asarray(ollama.Client(host="http://localhost:11434", trust_env=False).embeddings(model=EMBED_MODEL, prompt=query)["embedding"], dtype="float32")
q_vec = q_vec / (np.linalg.norm(q_vec) + 1e-12)
D, I = index.search(q_vec.reshape(1, -1), topk)

D.shape

(1, 5)

In [20]:

print(f"\nTop {topk} hits for query: '{query}'\n")
for rank, (dist, idx) in enumerate(zip(D[0], I[0]), start=1):
    chunk = chunks[idx]
    pretty_print(f"Rank {rank} | Score {dist:.4f} | {chunk['title']} (chunk {chunk['chunk_index']})")
    pretty_print("  ", chunk["preview"])
    print()


Top 5 hits for query: 'Best detective in the world'

Rank 1 | Score 0.3802 | Adventures of Sherlock Holmes (chunk 134)
   meet him at Boscombe Pool was someone who had been in Australia.” “What of
the rat, then?” Sherlock Holmes took a folded paper from his pocket and
flattened it out on the table. “This is a map of the Colony of Victoria,” he
said. “I wired to Bristol for it last night.” He put his hand over part of the
map. “What do you read?” “ARAT,” I read. “And now?” He raised his hand.
“BALLARAT.” “Quite so. Th

Rank 2 | Score 0.3763 | Adventures of Sherlock Holmes (chunk 216)
   I continue to retain the hat of the unknown gentleman who lost his Christmas
dinner.” “Did he not advertise?” “No.” “Then, what clue could you have as to his
identity?” “Only as much as we can deduce.” “From his hat?” “Precisely.” “But
you are joking. What can you gather from this old battered felt?” “Here is my
lens. You know my methods. What can you gather yourself as to the individuality
of the

Rank

In [21]:
# --- Simple windowed retrieval: expand each ANN hit with ±neighbor chunks ---


# Lookup map for (doc, chunk_index) -> global idx
KEY_TO_IDX = {(c["source_path"], c["chunk_index"]): gi for gi, c in enumerate(chunks)}


def expand_with_neighbors_simple(I, chunks, neighbors=1, max_out=8):
    """
    For each ANN hit in I[0], include the hit itself plus its ±neighbor
    chunks from the same document. Dedupe, preserve hit order, and stop
    once we've collected `max_out` chunks.

    Returns a flat list of context dicts — one per chunk (no merging).
    """
    seen = set()
    contexts = []

    for gi in I[0]:
        c = chunks[int(gi)]
        doc = c["source_path"]
        ci  = c["chunk_index"]
        # the hit itself + each neighbor
        for delta in range(-neighbors, neighbors + 1):
            j = KEY_TO_IDX.get((doc, ci + delta))
            if j is None or j in seen:
                continue
            seen.add(j)
            n = chunks[j]
            contexts.append({
                "title": n["title"],
                "source_path": n["source_path"],
                "chunk_index": n["chunk_index"],
                "text": n["text"],
                "approx_words": len(n["text"].split()),
            })
            if len(contexts) >= max_out:
                return contexts
    return contexts


def search_windowed(query: str, topk: int = TOPK, max_out: int = 8):
    """
    1) Embed the query (Ollama)
    2) ANN search the FAISS index for top-k chunks
    3) Decide how many of those hits to actually expand, using a simple
       confidence/margin heuristic on the FAISS scores
    4) Expand each chosen hit with ±neighbor chunks from the same document
    Returns: (df_hits, contexts)
    """
    q_vec = np.asarray(
        ollama.Client(host="http://localhost:11434", trust_env=False)
              .embeddings(model=EMBED_MODEL, prompt=query)["embedding"],
        dtype="float32",
    )
    q_vec = q_vec / (np.linalg.norm(q_vec) + 1e-12)
    D, I = index.search(q_vec.reshape(1, -1), topk)

    # ±1 neighbors for ~300-word chunks; widen if chunks are shorter
    neighbors = 1 if WORDS_PER_CHUNK >= 260 else 2

    # Confidence/margin heuristic: how many top hits should we actually expand?
    # - If the best match is strong (>= 0.35) AND clearly beats #2 (margin >= 0.05),
    #   we trust just that one hit.
    # - Otherwise the results look ambiguous, so cast a wider net (up to 3).
    top1   = float(D[0][0])
    top2   = float(D[0][1]) if len(D[0]) > 1 else 0.0
    margin = top1 - top2
    init_topk = 1 if (top1 >= 0.35 and margin >= 0.05) else min(3, topk)

    I_init = np.array([I[0][:init_topk]])

    contexts = expand_with_neighbors_simple(I_init, chunks, neighbors=neighbors, max_out=max_out)

    # Table of raw ANN hits (for transparency / debugging)
    rows = []
    for score, idx_ in zip(D[0].tolist(), I[0].tolist()):
        m = chunks[idx_]
        rows.append({
            "score": round(score, 3),
            "id": m["id"],
            "title": m["title"],
            "source_path": m["source_path"],
            "chunk_index": m["chunk_index"],
            "preview": (m["preview"][:220] + "…") if len(m["preview"]) > 220 else m["preview"],
        })
    df_hits = pd.DataFrame(rows)
    return df_hits, contexts


In [22]:
# --- Quick test query ---
QUERY = "Elizabeth Bennet's changing feelings for Mr. Darcy"
df_hits, contexts = search_windowed(QUERY, topk=TOPK, max_out=8)

print("=== Initial ANN Hits ===")
display(df_hits)

print("\n=== Expanded Contexts (hits + ±neighbors) ===")
display(pd.DataFrame([{
    "title": c["title"],
    "source": Path(c["source_path"]).name,
    "chunk_index": c["chunk_index"],
    "approx_words": c["approx_words"],
    "preview": (c["text"][:220] + "…") if len(c["text"]) > 220 else c["text"],
} for c in contexts]))

=== Initial ANN Hits ===


,score,id,title,source_path,chunk_index,preview
0,0.517,Pride_and_Prejudice#chunk457,Pride and Prejudice,corpus_jupyter/Pride_and_Prejudice.txt,457,"Mr. Darcy!--and so it does, I vow. Well, any f..."
1,0.509,Pride_and_Prejudice#chunk83,Pride and Prejudice,corpus_jupyter/Pride_and_Prejudice.txt,83,"employed, Elizabeth could not help observing, ..."
2,0.490,Pride_and_Prejudice#chunk518,Pride and Prejudice,corpus_jupyter/Pride_and_Prejudice.txt,518,good as a lord! And a special licence--you mus...
3,0.486,Pride_and_Prejudice#chunk349,Pride and Prejudice,corpus_jupyter/Pride_and_Prejudice.txt,349,they are! He takes them now for people of fash...
4,0.480,Pride_and_Prejudice#chunk357,Pride and Prejudice,corpus_jupyter/Pride_and_Prejudice.txt,357,who had expected to find in her as acute and u...



=== Expanded Contexts (hits + ±neighbors) ===


,title,source,chunk_index,approx_words,preview
0,Pride and Prejudice,Pride_and_Prejudice.txt,456,300,"but she does not know, no one can know, how mu..."
1,Pride and Prejudice,Pride_and_Prejudice.txt,457,300,"Mr. Darcy!--and so it does, I vow. Well, any f..."
2,Pride and Prejudice,Pride_and_Prejudice.txt,458,300,"again, was almost equal to what she had known ..."
3,Pride and Prejudice,Pride_and_Prejudice.txt,82,300,"Darcy were not such a great tall fellow, in co..."
4,Pride and Prejudice,Pride_and_Prejudice.txt,83,300,"employed, Elizabeth could not help observing, ..."
5,Pride and Prejudice,Pride_and_Prejudice.txt,84,300,to dance a reel at all; and now despise me if ...
6,Pride and Prejudice,Pride_and_Prejudice.txt,517,300,"utter a syllable. Nor was it under many, many ..."
7,Pride and Prejudice,Pride_and_Prejudice.txt,518,300,good as a lord! And a special licence--you mus...
